In [10]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [11]:
df = pd.read_csv("../data/raw/metadata.csv")

label_map = {
    'mel': 'melanoma',
    'nv': 'nevus',
    'bcc': 'basal cell carcinoma',
    'akiec': 'actinic keratosis',
    'bkl': 'benign keratosis'
}

df = df[df['dx'].isin(label_map.keys())]

df['label'] = df['dx'].map(label_map)

df['image_path'] = df['image_id'].apply(
    lambda x: "../data/raw/images/" + x + ".jpg"
)

print(df['label'].value_counts())

label
nevus                   6705
melanoma                1113
benign keratosis        1099
basal cell carcinoma     514
actinic keratosis        327
Name: count, dtype: int64


In [12]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df['label'],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['label'],
    random_state=42
)

In [13]:
IMG_SIZE = 160
BATCH_SIZE = 16

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    horizontal_flip=True,
    zoom_range=0.2
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    train_df,
    x_col='image_path',
    y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_dataframe(
    val_df,
    x_col='image_path',
    y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

Found 6830 validated image filenames belonging to 5 classes.
Found 1464 validated image filenames belonging to 5 classes.


In [14]:
print(train_generator.class_indices)

{'actinic keratosis': 0, 'basal cell carcinoma': 1, 'benign keratosis': 2, 'melanoma': 3, 'nevus': 4}


In [15]:
model = Sequential()

model.add(Conv2D(32, (3,3), activation='relu', input_shape=(160,160,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(2,2))

model.add(Conv2D(64, (3,3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(2,2))

model.add(Conv2D(128, (3,3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(2,2))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(5, activation='softmax'))

model.summary()

c:\Users\hadee\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 158, 158, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 158, 158, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 79, 79, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 77, 77, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 77, 77, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 38, 38, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 36, 36, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 36, 36, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 18, 18, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 41472)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     5,308,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,403,333 (20.61 MB)

 Trainable params: 5,402,885 (20.61 MB)

 Non-trainable params: 448 (1.75 KB)

In [16]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [17]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)

class_weights = dict(enumerate(class_weights))

print(class_weights)

{0: np.float64(5.965065502183406), 1: np.float64(3.7944444444444443), 2: np.float64(1.776332899869961), 3: np.float64(1.7535301668806162), 4: np.float64(0.29107180907734925)}


In [18]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3
)

In [19]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=12,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/12
427/427 ━━━━━━━━━━━━━━━━━━━━ 659s 2s/step - accuracy: 0.4691 - loss: 1.8697 - val_accuracy: 0.6025 - val_loss: 1.6079 - learning_rate: 1.0000e-04
Epoch 2/12
427/427 ━━━━━━━━━━━━━━━━━━━━ 594s 1s/step - accuracy: 0.5032 - loss: 1.4668 - val_accuracy: 0.5198 - val_loss: 1.1214 - learning_rate: 1.0000e-04
Epoch 3/12
427/427 ━━━━━━━━━━━━━━━━━━━━ 800s 2s/step - accuracy: 0.5070 - loss: 1.4689 - val_accuracy: 0.5649 - val_loss: 1.0253 - learning_rate: 1.0000e-04
Epoch 4/12
427/427 ━━━━━━━━━━━━━━━━━━━━ 877s 2s/step - accuracy: 0.5117 - loss: 1.3987 - val_accuracy: 0.5294 - val_loss: 1.0523 - learning_rate: 1.0000e-04
Epoch 5/12
427/427 ━━━━━━━━━━━━━━━━━━━━ 777s 2s/step - accuracy: 0.4865 - loss: 1.3810 - val_accuracy: 0.4638 - val_loss: 1.2123 - learning_rate: 1.0000e-04
Epoch 6/12
427/427 ━━━━━━━━━━━━━━━━━━━━ 770s 2s/step - accuracy: 0.4644 - loss: 1.3887 - val_accuracy: 0.5444 - val_loss: 0.9092 - learning_rate: 1.0000e-04
Epoch 7/12
427/427 ━━━━━━━━━━━━━━━━━━━━ 778s 2s/step - acc

In [20]:
model.save("../models/cnn_model_5class.h5")